# AOI — Train the real defect detector (DsPCBSD+ / YOLOv8) on Colab GPU

Trains the real YOLOv8 defect model that replaces the synthetic inference events.
See `docs/implementation/REAL_INFERENCE_INTEGRATION_PLAN.md` for the full plan.

**Runtime → Change runtime type → GPU (T4 is fine).**

Data-flow principle (important):
- **Dataset lives on Colab local disk** (`/content`) during training — mounted Drive is far too slow for 10k small-file reads.
- **Outputs (weights, runs, eval) live on Drive** via a symlink, so a disconnect never wipes them and `--resume` works.
- The big dataset **zip is cached on Drive** so you only download it once.

Do **not** upload the 10k images from your laptop — Colab pulls them from Kaggle far faster.

In [ ]:
# 1. Confirm GPU + mount Drive (Drive is for OUTPUTS and your own images only)
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/AOI'
os.makedirs(f'{DRIVE}/datasets', exist_ok=True)
os.makedirs(f'{DRIVE}/ml_models', exist_ok=True)
print('Drive ready at', DRIVE)

In [ ]:
# 2. Clone the repo (gives the variant-aware pipeline, not just plain YOLO) + deps
!rm -rf /content/AOI
!git clone https://github.com/lystiger/AOI.git /content/AOI
%cd /content/AOI
!pip install -q ultralytics kaggle

# Persist all training outputs to Drive so a disconnect can't wipe them and --resume works.
# (Only outputs go through Drive; the dataset stays on fast local disk below.)
!rm -rf /content/AOI/ml/models
!ln -s /content/drive/MyDrive/AOI/ml_models /content/AOI/ml/models
!ls -la /content/AOI/ml/models

### Kaggle token
Upload your `kaggle.json` (Kaggle → Account → Create New API Token) to `MyDrive/AOI/kaggle.json` once.
If the dataset slug 404s, find the current one with `!kaggle datasets list -s dspcbsd`.

In [ ]:
# 3. Get DsPCBSD+ onto LOCAL disk (cache the zip on Drive to avoid re-downloading)
import os, glob, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy2(f'{DRIVE}/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

zip_cache = f'{DRIVE}/datasets/dspcbsd_plus.zip'
if not os.path.exists(zip_cache):
    !kaggle datasets download -d enisteper1/dataset-of-pcb-surface-defects-dspcbsd -p /content/dl
    shutil.copy2(glob.glob('/content/dl/*.zip')[0], zip_cache)
    print('Cached zip to Drive.')
else:
    print('Using cached zip from Drive.')

!mkdir -p /content/data/dspcbsd_raw
!unzip -q -o "{zip_cache}" -d /content/data/dspcbsd_raw
print('Top-level contents of the download:')
!find /content/data/dspcbsd_raw -maxdepth 2 | head -40

In [ ]:
# 4. Normalize whatever layout the download has -> train/val/test + data.yaml (LOCAL disk)
!python -m ml.pipeline.dspcbsd_dataset \
    --source-root /content/data/dspcbsd_raw \
    --output-root /content/AOI/ml/data/dspcbsd_plus \
    --overwrite
print('\n--- data.yaml ---')
print(open('/content/AOI/ml/data/dspcbsd_plus/data.yaml').read())

### Training notes
- Repo defaults (`imgsz=1280, epochs=100, batch=8`) will OOM/time out a free T4. Start with `imgsz=640, epochs=60, batch=16`; raise `imgsz` later if the tiny trace defects need it.
- Outputs land under `ml/models/component_detection/` → which is symlinked to Drive, so they persist.
- If Colab disconnects mid-run, re-run the same cell with `--resume` appended to continue from `last.pt`.

In [ ]:
# 5a. Train the baseline YOLOv8s defect detector
!python -m ml.pipeline.component_train \
    --variant baseline \
    --dataset-root /content/AOI/ml/data/dspcbsd_plus \
    --imgsz 640 --epochs 60 --batch 16 --device 0

In [ ]:
# 5b. Train the channel-attention variant (feeds the S09 degradation comparison)
!python -m ml.pipeline.component_train \
    --variant channel_attention \
    --dataset-root /content/AOI/ml/data/dspcbsd_plus \
    --imgsz 640 --epochs 60 --batch 16 --device 0

In [ ]:
# 6. Evaluate on the held-out test split -> honest mAP/precision/recall + confusion matrix
!python -m ml.pipeline.component_evaluate \
    --variant baseline --split test \
    --dataset-root /content/AOI/ml/data/dspcbsd_plus \
    --imgsz 640

In [ ]:
# 7. Weights already persist on Drive (via the symlink). Confirm what landed.
!ls -la /content/drive/MyDrive/AOI/ml_models/component_detection/*.pt 2>/dev/null || echo 'no top-level weights yet'
!find /content/drive/MyDrive/AOI/ml_models -name 'best*.pt' | head

## Next steps
1. Download `best-baseline.pt` (and `best.pt`) from `MyDrive/AOI/ml_models/component_detection/` back to your laptop into `ml/models/defect_detection/`.
2. Back in the app, the wired `inference_runner` (Workstream 3) loads it and POSTs real events into the Loki/Grafana pipeline.
3. Run the experiment suite against the real model per the test matrix in the plan.